In [47]:
import pandas as pd
from pathlib import Path

In [48]:
silver_path = Path("../data/silver")
gold_path = Path("../data/gold")

gold_path.mkdir(parents=True, exist_ok=True)

# Cart Details

In [81]:
cart_details = pd.read_parquet(
    silver_path / "cart_details.parquet"
)

print(cart_details.info())

<class 'pandas.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   cart_item_id         800 non-null    int64  
 1   cart_id              800 non-null    int64  
 2   user_id              800 non-null    int64  
 3   product_id           800 non-null    int64  
 4   title                800 non-null    str    
 5   price                800 non-null    float64
 6   quantity             800 non-null    int64  
 7   total                800 non-null    float64
 8   discount_percentage  800 non-null    float64
 9   discounted_total     800 non-null    float64
 10  thumbnail            800 non-null    str    
dtypes: float64(4), int64(5), str(2)
memory usage: 149.2 KB
None


In [82]:
fact_cart_items = cart_details[
    [
        "cart_item_id",
        "cart_id",
        "product_id",
        "user_id",
        "quantity",
        "price",
        "total",
        "discount_percentage",
        "discounted_total"
    ]
].copy()

fact_cart_items = fact_cart_items.rename(columns={
    "price": "unit_price",
    "total": "gross_amount",
    "discounted_total": "net_amount"
})

fact_cart_items.head()

,cart_item_id,cart_id,product_id,user_id,quantity,unit_price,gross_amount,discount_percentage,net_amount
0,1,1,162,1,4,29.99,119.96,12.13,105.41
1,2,1,113,1,3,3999.99,11999.97,12.10,10547.97
2,3,1,122,1,3,299.99,899.97,6.69,839.76
3,4,1,138,1,2,8.99,17.98,1.71,17.67
4,5,2,86,2,5,19.99,99.95,6.83,93.12


In [83]:
fact_cart_items["discount_amount"] = (
    fact_cart_items["gross_amount"]
    - fact_cart_items["net_amount"]
)

fact_cart_items.head()

,cart_item_id,cart_id,product_id,user_id,quantity,unit_price,gross_amount,discount_percentage,net_amount,discount_amount
0,1,1,162,1,4,29.99,119.96,12.13,105.41,14.55
1,2,1,113,1,3,3999.99,11999.97,12.10,10547.97,1452.00
2,3,1,122,1,3,299.99,899.97,6.69,839.76,60.21
3,4,1,138,1,2,8.99,17.98,1.71,17.67,0.31
4,5,2,86,2,5,19.99,99.95,6.83,93.12,6.83


In [84]:
fact_cart_items["discount_amount"].describe()

count      800.000000
mean       471.961313
std       2260.159338
min          0.000000
25%          2.690000
50%          9.790000
75%         60.210000
max      30413.990000
Name: discount_amount, dtype: float64

In [85]:
print(
    "Descuentos negativos:",
    (fact_cart_items["discount_amount"] < 0).sum()
)

print(
    "Importes netos superiores a los brutos:",
    (
        fact_cart_items["net_amount"]
        > fact_cart_items["gross_amount"]
    ).sum()
)

print(
    "Diferencias entre el descuento calculado y el porcentaje:",
    (
        (
            fact_cart_items["gross_amount"]
            * fact_cart_items["discount_percentage"] / 100
        )
        - fact_cart_items["discount_amount"]
    ).abs().gt(0.02).sum()
)

Descuentos negativos: 0
Importes netos superiores a los brutos: 0
Diferencias entre el descuento calculado y el porcentaje: 0


In [86]:
carts_silver = pd.read_parquet(silver_path / "carts.parquet")
products_silver = pd.read_parquet(silver_path / "products.parquet")
dim_users = pd.read_parquet(silver_path / "dim_users.parquet")

In [87]:
print(
    "Duplicados cart_item_id:",
    fact_cart_items["cart_item_id"].duplicated().sum()
)

print(
    "Claves nulas:",
    fact_cart_items[
        ["cart_item_id", "cart_id", "product_id", "user_id"]
    ].isna().sum().sum()
)

print(
    "Carritos inexistentes:",
    (~fact_cart_items["cart_id"].isin(carts_silver["cart_id"])).sum()
)

print(
    "Productos inexistentes:",
    (~fact_cart_items["product_id"].isin(products_silver["product_id"])).sum()
)

print(
    "Usuarios inexistentes:",
    (~fact_cart_items["user_id"].isin(dim_users["user_id"])).sum()
)

Duplicados cart_item_id: 0
Claves nulas: 0
Carritos inexistentes: 0
Productos inexistentes: 0
Usuarios inexistentes: 0


In [88]:
print("Filas:", len(fact_cart_items))
print("Columnas:", len(fact_cart_items.columns))

display(fact_cart_items.head())

fact_cart_items.info()

Filas: 800
Columnas: 10


,cart_item_id,cart_id,product_id,user_id,quantity,unit_price,gross_amount,discount_percentage,net_amount,discount_amount
0,1,1,162,1,4,29.99,119.96,12.13,105.41,14.55
1,2,1,113,1,3,3999.99,11999.97,12.10,10547.97,1452.00
2,3,1,122,1,3,299.99,899.97,6.69,839.76,60.21
3,4,1,138,1,2,8.99,17.98,1.71,17.67,0.31
4,5,2,86,2,5,19.99,99.95,6.83,93.12,6.83


<class 'pandas.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   cart_item_id         800 non-null    int64  
 1   cart_id              800 non-null    int64  
 2   product_id           800 non-null    int64  
 3   user_id              800 non-null    int64  
 4   quantity             800 non-null    int64  
 5   unit_price           800 non-null    float64
 6   gross_amount         800 non-null    float64
 7   discount_percentage  800 non-null    float64
 8   net_amount           800 non-null    float64
 9   discount_amount      800 non-null    float64
dtypes: float64(5), int64(5)
memory usage: 62.6 KB


In [89]:
fact_cart_items.to_parquet(
    gold_path / "fact_cart_items.parquet",
    index=False
)

In [90]:
fact_cart_items_gold = pd.read_parquet(
    gold_path / "fact_cart_items.parquet"
)

print(fact_cart_items_gold.shape)

(800, 10)


# Products

In [59]:
products = pd.read_parquet(
    silver_path / "products.parquet"
)

print(products.info())

<class 'pandas.DataFrame'>
RangeIndex: 194 entries, 0 to 193
Data columns (total 21 columns):
 #   Column                  Non-Null Count  Dtype              
---  ------                  --------------  -----              
 0   product_id              194 non-null    int64              
 1   title                   194 non-null    str                
 2   description             194 non-null    str                
 3   category                194 non-null    str                
 4   brand                   102 non-null    str                
 5   price                   194 non-null    float64            
 6   discount_percentage     194 non-null    float64            
 7   rating                  194 non-null    float64            
 8   stock                   194 non-null    int64              
 9   sku                     194 non-null    str                
 10  weight                  194 non-null    int64              
 11  minimum_order_quantity  194 non-null    int64           

In [60]:
product_columns = [
    "product_id",
    "title",
    "category",
    "brand",
    "price",
    "discount_percentage",
    "rating",
    "stock",
    "availability_status",
    "weight",
    "minimum_order_quantity",
    "warranty_information",
    "shipping_information",
    "return_policy",
    "created_at",
    "updated_at"
]

dim_product = products[product_columns].copy()

dim_product.head()

,product_id,title,category,brand,price,discount_percentage,rating,stock,availability_status,weight,minimum_order_quantity,warranty_information,shipping_information,return_policy,created_at,updated_at
0,1,Essence Mascara Lash Princess,beauty,Essence,9.99,10.48,2.56,99,In Stock,4,48,1 week warranty,Ships in 3-5 business days,No return policy,2025-10-09 14:47:01.588+00:00,2026-05-23 11:27:41.868+00:00
1,2,Eyeshadow Palette with Mirror,beauty,Glamour Beauty,19.99,18.19,2.86,34,In Stock,9,20,1 year warranty,Ships in 2 weeks,7 days return policy,2024-12-03 11:29:45.706+00:00,2025-08-24 16:52:47.806+00:00
2,3,Powder Canister,beauty,Velvet Touch,14.99,9.84,4.64,89,In Stock,8,22,3 months warranty,Ships in 1-2 business days,No return policy,2025-01-08 18:30:12.844+00:00,2025-11-26 14:49:26.980+00:00
3,4,Red Lipstick,beauty,Chic Cosmetics,12.99,12.16,4.36,91,In Stock,1,40,3 year warranty,Ships in 1 week,7 days return policy,2025-11-22 23:28:00.911+00:00,2026-01-26 01:31:40.721+00:00
4,5,Red Nail Polish,beauty,Nail Couture,8.99,11.44,4.32,79,In Stock,8,22,1 month warranty,Ships overnight,No return policy,2025-10-03 07:39:53.470+00:00,2026-04-24 00:21:44.635+00:00


In [61]:
dim_product["brand"] = (
    dim_product["brand"]
    .fillna("Unknown")
)

print(dim_product["brand"].value_counts().head(10))

print(
    "\nValores nulos:",
    dim_product["brand"].isna().sum()
)



brand
Unknown           92
Apple             14
Rolex              6
Samsung            5
Fashion Shades     4
Oppo               3
Realme             3
Vivo               3
Dodge              3
Calvin Klein       2
Name: count, dtype: int64

Valores nulos: 0


In [62]:
print(
    "Productos duplicados:",
    dim_product["product_id"].duplicated().sum()
)

print(
    "Claves nulas:",
    dim_product["product_id"].isna().sum()
)

print(
    "Productos sin categoría:",
    dim_product["category"].isna().sum()
)

print(
    "Precios negativos:",
    (dim_product["price"] < 0).sum()
)

print(
    "Stock negativo:",
    (dim_product["stock"] < 0).sum()
)

print(
    "Calificaciones fuera de rango:",
    (
        (dim_product["rating"] < 0)
        | (dim_product["rating"] > 5)
    ).sum()
)

Productos duplicados: 0
Claves nulas: 0
Productos sin categoría: 0
Precios negativos: 0
Stock negativo: 0
Calificaciones fuera de rango: 0


In [63]:
print(
    "Productos de fact_cart_items inexistentes en dim_product:",
    (
        ~fact_cart_items["product_id"].isin(
            dim_product["product_id"]
        )
    ).sum()
)

Productos de fact_cart_items inexistentes en dim_product: 0


In [64]:
dim_product.to_parquet(
    gold_path / "dim_product.parquet",
    index=False
)

# Dim_User

In [65]:
users = pd.read_parquet(
    silver_path / "dim_users.parquet"
)

print(users.info())

<class 'pandas.DataFrame'>
RangeIndex: 208 entries, 0 to 207
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   user_id      208 non-null    int64
 1   age          208 non-null    int64
 2   gender       208 non-null    str  
 3   role         208 non-null    str  
 4   city         208 non-null    str  
 5   state        208 non-null    str  
 6   state_code   208 non-null    str  
 7   postal_code  208 non-null    str  
 8   country      208 non-null    str  
dtypes: int64(2), str(7)
memory usage: 24.3 KB
None


In [66]:
print("Distribución por género:")
print(users["gender"].value_counts())

print("\nDistribución por rol:")
print(users["role"].value_counts())

print("\nDistribución por país:")
print(users["country"].value_counts())

print("\nCantidad de ciudades:", users["city"].nunique())
print("Cantidad de estados:", users["state"].nunique())

print("\nDistribución de edades:")
print(users["age"].describe())

Distribución por género:
gender
female    106
male      102
Name: count, dtype: int64

Distribución por rol:
role
user         193
moderator     10
admin          5
Name: count, dtype: int64

Distribución por país:
country
United States    208
Name: count, dtype: int64

Cantidad de ciudades: 20
Cantidad de estados: 49

Distribución de edades:
count    208.000000
mean      33.139423
std        5.544269
min       23.000000
25%       29.000000
50%       32.000000
75%       36.000000
max       46.000000
Name: age, dtype: float64


In [67]:
user_columns = [
    "user_id",
    "age",
    "gender",
    "city",
    "state",
    "state_code",
    "postal_code",
    "country"
]

dim_user = users[user_columns].copy()

dim_user.head()

,user_id,age,gender,city,state,state_code,postal_code,country
0,1,29,female,Phoenix,Mississippi,MS,29112,United States
1,2,36,male,Houston,Alabama,AL,38807,United States
2,3,43,female,Washington,Alabama,AL,32822,United States
3,4,46,male,Seattle,Pennsylvania,PA,68354,United States
4,5,31,female,Jacksonville,Colorado,CO,26593,United States


In [68]:
print(
    "Usuarios duplicados:",
    dim_user["user_id"].duplicated().sum()
)

print(
    "Claves nulas:",
    dim_user["user_id"].isna().sum()
)

print(
    "Usuarios sin ciudad:",
    dim_user["city"].isna().sum()
)

print(
    "Usuarios sin estado:",
    dim_user["state"].isna().sum()
)

print(
    "Edades inválidas:",
    (
        (dim_user["age"] <= 0)
        | (dim_user["age"] > 120)
    ).sum()
)

print(
    "Géneros no reconocidos:",
    (~dim_user["gender"].isin(["female", "male"])).sum()
)

Usuarios duplicados: 0
Claves nulas: 0
Usuarios sin ciudad: 0
Usuarios sin estado: 0
Edades inválidas: 0
Géneros no reconocidos: 0


In [69]:
print(
    "Usuarios de fact_cart_items inexistentes en dim_user:",
    (
        ~fact_cart_items["user_id"].isin(
            dim_user["user_id"]
        )
    ).sum()
)

Usuarios de fact_cart_items inexistentes en dim_user: 0


In [70]:
dim_user.to_parquet(
    gold_path / "dim_user.parquet",
    index=False
)

dim_user_gold = pd.read_parquet(
    gold_path / "dim_user.parquet"
)

print(dim_user_gold.shape)

(208, 8)


# Fact Carts

In [71]:
carts = pd.read_parquet(
    silver_path / "carts.parquet"
)

print(carts.info())

carts.head()

<class 'pandas.DataFrame'>
RangeIndex: 208 entries, 0 to 207
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   cart_id           208 non-null    int64  
 1   user_id           208 non-null    int64  
 2   total             208 non-null    float64
 3   discounted_total  208 non-null    float64
 4   total_products    208 non-null    int64  
 5   total_quantity    208 non-null    int64  
dtypes: float64(2), int64(4)
memory usage: 9.9 KB
None


,cart_id,user_id,total,discounted_total,total_products,total_quantity
0,1,1,13037.88,11510.81,4,12
1,2,2,139.93,125.70,2,7
2,3,3,1794.85,1625.77,6,15
3,4,4,689.93,636.81,2,7
4,5,5,1467.88,1205.80,3,12


In [91]:
fact_carts = carts[
    [
        "cart_id",
        "user_id",
        "total",
        "discounted_total",
        "total_products",
        "total_quantity"
    ]
].copy()

fact_carts = fact_carts.rename(columns={
    "total": "gross_amount",
    "discounted_total": "net_amount"
})

fact_carts["discount_amount"] = (
    fact_carts["gross_amount"]
    - fact_carts["net_amount"]
)

fact_carts.head()

,cart_id,user_id,gross_amount,net_amount,total_products,total_quantity,discount_amount
0,1,1,13037.88,11510.81,4,12,1527.07
1,2,2,139.93,125.70,2,7,14.23
2,3,3,1794.85,1625.77,6,15,169.08
3,4,4,689.93,636.81,2,7,53.12
4,5,5,1467.88,1205.80,3,12,262.08


In [92]:
fact_carts[
    [
        "gross_amount",
        "net_amount",
        "discount_amount",
        "total_products",
        "total_quantity"
    ]
].describe()

,gross_amount,net_amount,discount_amount,total_products,total_quantity
count,208.000000,208.000000,208.000000,208.000000,208.000000
mean,18434.031875,16618.796058,1815.235817,3.846154,11.620192
std,34725.549339,31041.763712,4303.513812,1.492456,5.425381
min,6.970000,6.650000,0.100000,2.000000,2.000000
25%,400.677500,360.267500,43.987500,2.000000,7.000000
50%,1994.385000,1650.155000,233.880000,4.000000,12.000000
75%,18388.625000,17121.535000,1595.197500,5.000000,16.000000
max,186110.850000,155507.160000,30603.690000,6.000000,27.000000


In [93]:
print(
    "Carritos duplicados:",
    fact_carts["cart_id"].duplicated().sum()
)

print(
    "Claves nulas:",
    fact_carts["cart_id"].isna().sum()
)

print(
    "Importes negativos:",
    (
        (fact_carts["gross_amount"] < 0)
        | (fact_carts["net_amount"] < 0)
    ).sum()
)

print(
    "Descuentos negativos:",
    (fact_carts["discount_amount"] < 0).sum()
)

print(
    "Carritos con net_amount superior a gross_amount:",
    (
        fact_carts["net_amount"]
        > fact_carts["gross_amount"]
    ).sum()
)

Carritos duplicados: 0
Claves nulas: 0
Importes negativos: 0
Descuentos negativos: 0
Carritos con net_amount superior a gross_amount: 0


In [94]:
print(
    "Usuarios inexistentes en dim_user:",
    (
        ~fact_carts["user_id"].isin(
            dim_user["user_id"]
        )
    ).sum()
)

Usuarios inexistentes en dim_user: 0


In [95]:
cart_items_summary = (
    fact_cart_items
    .groupby("cart_id")
    .agg(
        gross_amount_items=("gross_amount", "sum"),
        net_amount_items=("net_amount", "sum"),
        total_products_items=("cart_item_id", "count"),
        total_quantity_items=("quantity", "sum")
    )
    .reset_index()
)

cart_comparison = fact_carts.merge(
    cart_items_summary,
    on="cart_id",
    how="left"
)

cart_comparison.head()

,cart_id,user_id,gross_amount,net_amount,total_products,total_quantity,discount_amount,gross_amount_items,net_amount_items,total_products_items,total_quantity_items
0,1,1,13037.88,11510.81,4,12,1527.07,13037.88,11510.81,4,12
1,2,2,139.93,125.70,2,7,14.23,139.93,125.70,2,7
2,3,3,1794.85,1625.77,6,15,169.08,1794.85,1625.77,6,15
3,4,4,689.93,636.81,2,7,53.12,689.93,636.81,2,7
4,5,5,1467.88,1205.80,3,12,262.08,1467.88,1205.80,3,12


In [96]:
print(
    "Diferencias en importe bruto:",
    (
        (cart_comparison["gross_amount"]
         - cart_comparison["gross_amount_items"]).abs() > 0.02
    ).sum()
)

print(
    "Diferencias en importe neto:",
    (
        (cart_comparison["net_amount"]
         - cart_comparison["net_amount_items"]).abs() > 0.02
    ).sum()
)

print(
    "Diferencias en cantidad de productos:",
    (
        cart_comparison["total_products"]
        != cart_comparison["total_products_items"]
    ).sum()
)

print(
    "Diferencias en cantidad de unidades:",
    (
        cart_comparison["total_quantity"]
        != cart_comparison["total_quantity_items"]
    ).sum()
)

Diferencias en importe bruto: 0
Diferencias en importe neto: 0
Diferencias en cantidad de productos: 0
Diferencias en cantidad de unidades: 0


In [100]:
fact_cart_items.to_parquet(
    gold_path / "fact_cart_items.parquet",
    index=False
)

fact_carts.to_parquet(
    gold_path / "fact_carts.parquet",
    index=False
)

print("Tablas de hechos guardadas correctamente.")

Tablas de hechos guardadas correctamente.


# Fact Product reviews

In [105]:
product_reviews = pd.read_parquet(
    silver_path / "product_reviews.parquet"
)

print("Dimensiones de origen:", product_reviews.shape)
print("Columnas:", product_reviews.columns.tolist())

Dimensiones de origen: (582, 6)
Columnas: ['review_id', 'product_id', 'reviewer_id', 'rating', 'comment', 'review_date']


In [108]:
reviewer_identity = pd.read_parquet(
    silver_path / "reviewer_identity.parquet"
)

print("Dimensiones:", reviewer_identity.shape)
print("Columnas:", reviewer_identity.columns.tolist())

Dimensiones: (191, 3)
Columnas: ['reviewer_id', 'reviewer_name', 'reviewer_email']


In [106]:
fact_product_reviews = product_reviews[
    [
        "review_id",
        "product_id",
        "reviewer_id",
        "rating",
        "review_date"
    ]
].copy()

In [110]:
print("Dimensiones:", fact_product_reviews.shape)

print("\nValores nulos:")
print(fact_product_reviews.isnull().sum())

print(
    "\nIDs de reseña duplicados:",
    fact_product_reviews["review_id"].duplicated().sum()
)

print(
    "\nCalificaciones fuera de rango:",
    (~fact_product_reviews["rating"].between(1, 5)).sum()
)

print(
    "\nProductos inexistentes:",
    (~fact_product_reviews["product_id"].isin(
        dim_product["product_id"]
    )).sum()
)

print(
    "Reseñadores inexistentes:",
    (~fact_product_reviews["reviewer_id"].isin(
        reviewer_identity["reviewer_id"]
    )).sum()
)

Dimensiones: (582, 5)

Valores nulos:
review_id      0
product_id     0
reviewer_id    0
rating         0
review_date    0
dtype: int64

IDs de reseña duplicados: 0

Calificaciones fuera de rango: 0

Productos inexistentes: 0
Reseñadores inexistentes: 0


In [111]:
print("Fecha mínima:", fact_product_reviews["review_date"].min())
print("Fecha máxima:", fact_product_reviews["review_date"].max())

print(
    "Fechas únicas:",
    fact_product_reviews["review_date"].nunique()
)


Fecha mínima: 2025-04-30 09:41:02.053000+00:00
Fecha máxima: 2025-04-30 09:41:02.054000+00:00
Fechas únicas: 2


In [114]:
fact_product_reviews["review_date"].value_counts()


review_date
2025-04-30 09:41:02.053+00:00    351
2025-04-30 09:41:02.054+00:00    231
Name: count, dtype: int64

In [115]:
fact_product_reviews["review_date"].dt.date.value_counts()

review_date
2025-04-30    582
Name: count, dtype: int64

In [116]:
dim_date = pd.DataFrame({
    "date": pd.to_datetime(
        fact_product_reviews["review_date"].dt.date.unique()
    )
})

dim_date["date_key"] = (
    dim_date["date"].dt.strftime("%Y%m%d").astype("int64")
)

dim_date["year"] = dim_date["date"].dt.year
dim_date["quarter"] = dim_date["date"].dt.quarter
dim_date["month"] = dim_date["date"].dt.month
dim_date["month_name"] = dim_date["date"].dt.month_name()
dim_date["day"] = dim_date["date"].dt.day
dim_date["day_of_week"] = dim_date["date"].dt.dayofweek
dim_date["day_name"] = dim_date["date"].dt.day_name()

C:\Users\macro\AppData\Local\Temp\ipykernel_16904\582002128.py:16: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  dim_date["day_of_week"] = dim_date["date"].dt.dayofweek


In [117]:
print("Dimensiones:", dim_date.shape)
display(dim_date)

Dimensiones: (1, 9)


,date,date_key,year,quarter,month,month_name,day,day_of_week,day_name
0,2025-04-30,20250430,2025,2,4,April,30,2,Wednesday


In [118]:
fact_product_reviews["date"] = (
    fact_product_reviews["review_date"].dt.date
)

fact_product_reviews["date"] = pd.to_datetime(
    fact_product_reviews["date"]
)

In [119]:
fact_product_reviews = fact_product_reviews.merge(
    dim_date[["date", "date_key"]],
    on="date",
    how="left",
    validate="many_to_one"
)

In [120]:
print("Valores nulos en date_key:",
      fact_product_reviews["date_key"].isnull().sum())

print("Fechas sin correspondencia:",
      (~fact_product_reviews["date"].isin(dim_date["date"])).sum())

print("Claves de fecha únicas:",
      fact_product_reviews["date_key"].nunique())

Valores nulos en date_key: 0
Fechas sin correspondencia: 0
Claves de fecha únicas: 1


In [121]:
fact_product_reviews = fact_product_reviews.drop(
    columns=["date", "review_date"]
)

In [122]:
print("Dimensiones:", fact_product_reviews.shape)
print("Columnas:", fact_product_reviews.columns.tolist())

display(fact_product_reviews.head())

Dimensiones: (582, 5)
Columnas: ['review_id', 'product_id', 'reviewer_id', 'rating', 'date_key']


,review_id,product_id,reviewer_id,rating,date_key
0,1,1,1,3,20250430
1,2,1,2,4,20250430
2,3,1,1,5,20250430
3,4,2,3,5,20250430
4,5,2,4,4,20250430


In [123]:
fact_product_reviews.to_parquet(
    gold_path / "fact_product_reviews.parquet",
    index=False
)

dim_date.to_parquet(
    gold_path / "dim_date.parquet",
    index=False
)

print("fact_product_reviews y dim_date guardadas correctamente.")

fact_product_reviews y dim_date guardadas correctamente.


# Dim Reviewer

In [124]:
dim_reviewer = reviewer_identity[
    ["reviewer_id"]
].copy()

In [125]:
print("Dimensiones:", dim_reviewer.shape)

print(
    "IDs duplicados:",
    dim_reviewer["reviewer_id"].duplicated().sum()
)

print(
    "IDs nulos:",
    dim_reviewer["reviewer_id"].isnull().sum()
)

print(
    "Reseñadores sin correspondencia:",
    (~fact_product_reviews["reviewer_id"].isin(
        dim_reviewer["reviewer_id"]
    )).sum()
)

Dimensiones: (191, 1)
IDs duplicados: 0
IDs nulos: 0
Reseñadores sin correspondencia: 0


In [126]:
dim_reviewer.to_parquet(
    gold_path / "dim_reviewer.parquet",
    index=False
)

# Bridge products tags

In [128]:
product_tags = pd.read_parquet(
    silver_path / "product_tags.parquet"
)

print("Dimensiones:", product_tags.shape)
print("Columnas:", product_tags.columns.tolist())

display(product_tags.head(10))

Dimensiones: (364, 2)
Columnas: ['product_id', 'tag']


,product_id,tag
0,1,beauty
1,1,mascara
2,2,beauty
3,2,eyeshadow
4,3,beauty
5,3,face powder
6,4,beauty
7,4,lipstick
8,5,beauty
9,5,nail polish


In [129]:
bridge_product_tags = product_tags.copy()

In [130]:
print("Dimensiones:", bridge_product_tags.shape)

print(
    "Combinaciones producto-etiqueta duplicadas:",
    bridge_product_tags.duplicated(
        subset=["product_id", "tag"]
    ).sum()
)

print(
    "Product IDs nulos:",
    bridge_product_tags["product_id"].isnull().sum()
)

print(
    "Etiquetas nulas:",
    bridge_product_tags["tag"].isnull().sum()
)

print(
    "Productos inexistentes:",
    (~bridge_product_tags["product_id"].isin(
        dim_product["product_id"]
    )).sum()
)

Dimensiones: (364, 2)
Combinaciones producto-etiqueta duplicadas: 0
Product IDs nulos: 0
Etiquetas nulas: 0
Productos inexistentes: 0


In [131]:
print(
    "Etiquetas únicas:",
    bridge_product_tags["tag"].nunique()
)

print(
    "Productos asociados a etiquetas:",
    bridge_product_tags["product_id"].nunique()
)

print(
    "Promedio de etiquetas por producto:",
    round(
        bridge_product_tags.groupby("product_id")["tag"].count().mean(),
        2
    )
)

Etiquetas únicas: 138
Productos asociados a etiquetas: 194
Promedio de etiquetas por producto: 1.88


In [132]:
bridge_product_tags.to_parquet(
    gold_path / "bridge_product_tags.parquet",
    index=False
)

print("bridge_product_tags guardada correctamente.")

bridge_product_tags guardada correctamente.


In [133]:
gold_files = sorted(gold_path.glob("*.parquet"))

for file in gold_files:
    df = pd.read_parquet(file)

    print(
        f"{file.name:<35} "
        f"Filas: {len(df):<5} "
        f"Columnas: {len(df.columns)}"
    )

bridge_product_tags.parquet         Filas: 364   Columnas: 2
dim_date.parquet                    Filas: 1     Columnas: 9
dim_product.parquet                 Filas: 194   Columnas: 16
dim_reviewer.parquet                Filas: 191   Columnas: 1
dim_user.parquet                    Filas: 208   Columnas: 8
fact_cart_items.parquet             Filas: 800   Columnas: 10
fact_carts.parquet                  Filas: 208   Columnas: 7
fact_product_reviews.parquet        Filas: 582   Columnas: 5


In [134]:
checks = {
    "fact_cart_items → dim_product": (
        fact_cart_items["product_id"].isin(
            dim_product["product_id"]
        ).all()
    ),

    "fact_cart_items → dim_user": (
        fact_cart_items["user_id"].isin(
            dim_user["user_id"]
        ).all()
    ),

    "fact_carts → dim_user": (
        fact_carts["user_id"].isin(
            dim_user["user_id"]
        ).all()
    ),

    "fact_product_reviews → dim_product": (
        fact_product_reviews["product_id"].isin(
            dim_product["product_id"]
        ).all()
    ),

    "fact_product_reviews → dim_reviewer": (
        fact_product_reviews["reviewer_id"].isin(
            dim_reviewer["reviewer_id"]
        ).all()
    ),

    "fact_product_reviews → dim_date": (
        fact_product_reviews["date_key"].isin(
            dim_date["date_key"]
        ).all()
    ),

    "bridge_product_tags → dim_product": (
        bridge_product_tags["product_id"].isin(
            dim_product["product_id"]
        ).all()
    )
}

for relationship, result in checks.items():
    print(
        f"{relationship}:",
        "OK" if result else "ERROR"
    )

fact_cart_items → dim_product: OK
fact_cart_items → dim_user: OK
fact_carts → dim_user: OK
fact_product_reviews → dim_product: OK
fact_product_reviews → dim_reviewer: OK
fact_product_reviews → dim_date: OK
bridge_product_tags → dim_product: OK


In [135]:
primary_key_checks = {
    "fact_cart_items": (
        fact_cart_items, ["cart_item_id"]
    ),
    "fact_carts": (
        fact_carts, ["cart_id"]
    ),
    "fact_product_reviews": (
        fact_product_reviews, ["review_id"]
    ),
    "dim_product": (
        dim_product, ["product_id"]
    ),
    "dim_user": (
        dim_user, ["user_id"]
    ),
    "dim_reviewer": (
        dim_reviewer, ["reviewer_id"]
    ),
    "dim_date": (
        dim_date, ["date_key"]
    ),
    "bridge_product_tags": (
        bridge_product_tags, ["product_id", "tag"]
    )
}

for table_name, (df, key_columns) in primary_key_checks.items():

    nulls = df[key_columns].isnull().any(axis=1).sum()

    duplicates = df.duplicated(
        subset=key_columns
    ).sum()

    print(
        f"{table_name}: "
        f"nulos={nulls}, "
        f"duplicados={duplicates}"
    )

fact_cart_items: nulos=0, duplicados=0
fact_carts: nulos=0, duplicados=0
fact_product_reviews: nulos=0, duplicados=0
dim_product: nulos=0, duplicados=0
dim_user: nulos=0, duplicados=0
dim_reviewer: nulos=0, duplicados=0
dim_date: nulos=0, duplicados=0
bridge_product_tags: nulos=0, duplicados=0


In [136]:
fact_cart_items[
    [
        "quantity",
        "unit_price",
        "gross_amount",
        "discount_percentage",
        "net_amount",
        "discount_amount"
    ]
].describe().round(2)

,quantity,unit_price,gross_amount,discount_percentage,net_amount,discount_amount
count,800.00,800.00,800.00,800.00,800.00,800.00
mean,3.02,1558.96,4792.85,10.60,4320.89,471.96
std,1.41,5414.43,18490.88,5.36,16544.45,2260.16
min,1.00,0.79,0.99,0.04,0.84,0.00
25%,2.00,12.99,29.98,6.54,26.15,2.69
50%,3.00,34.99,99.95,11.05,86.93,9.79
75%,4.00,299.99,519.96,14.87,451.46,60.21
max,5.00,36999.99,184999.95,19.61,154585.96,30413.99


In [137]:
total_metrics = {
    "total_cart_lines": len(fact_cart_items),

    "total_units": fact_cart_items["quantity"].sum(),

    "total_gross_amount": fact_cart_items["gross_amount"].sum(),

    "total_net_amount": fact_cart_items["net_amount"].sum(),

    "total_discount_amount": fact_cart_items["discount_amount"].sum(),

    "average_discount_percentage": (
        fact_cart_items["discount_percentage"].mean()
    ),

    "effective_discount_percentage": (
        fact_cart_items["discount_amount"].sum()
        / fact_cart_items["gross_amount"].sum()
        * 100
    )
}

for metric, value in total_metrics.items():
    print(f"{metric}: {value:,.2f}")

total_cart_lines: 800.00
total_units: 2,417.00
total_gross_amount: 3,834,278.63
total_net_amount: 3,456,709.58
total_discount_amount: 377,569.05
average_discount_percentage: 10.60
effective_discount_percentage: 9.85


In [138]:
print(
    "Diferencia importe bruto:",
    round(
        fact_cart_items["gross_amount"].sum()
        - fact_carts["gross_amount"].sum(),
        2
    )
)

print(
    "Diferencia importe neto:",
    round(
        fact_cart_items["net_amount"].sum()
        - fact_carts["net_amount"].sum(),
        2
    )
)

print(
    "Diferencia descuentos:",
    round(
        fact_cart_items["discount_amount"].sum()
        - fact_carts["discount_amount"].sum(),
        2
    )
)

print(
    "Diferencia unidades:",
    fact_cart_items["quantity"].sum()
    - fact_carts["total_quantity"].sum()
)

Diferencia importe bruto: 0.0
Diferencia importe neto: 0.0
Diferencia descuentos: 0.0
Diferencia unidades: 0


# Conectar con sqlite

In [139]:
import sqlite3
from pathlib import Path

# Ruta de la base de datos
db_path = Path("../data/database/markethub.db")

# Crear conexión
conn = sqlite3.connect(db_path)

print("Conexión establecida correctamente.")

Conexión establecida correctamente.


In [140]:
gold_tables = {
    "fact_cart_items": fact_cart_items,
    "fact_carts": fact_carts,
    "fact_product_reviews": fact_product_reviews,
    "dim_product": dim_product,
    "dim_user": dim_user,
    "dim_reviewer": dim_reviewer,
    "dim_date": dim_date,
    "bridge_product_tags": bridge_product_tags
}

for table_name, dataframe in gold_tables.items():
    dataframe.to_sql(
        name=table_name,
        con=conn,
        if_exists="replace",
        index=False
    )

    print(f"Tabla cargada: {table_name} | Registros: {len(dataframe)}")

Tabla cargada: fact_cart_items | Registros: 800
Tabla cargada: fact_carts | Registros: 208
Tabla cargada: fact_product_reviews | Registros: 582
Tabla cargada: dim_product | Registros: 194
Tabla cargada: dim_user | Registros: 208
Tabla cargada: dim_reviewer | Registros: 191
Tabla cargada: dim_date | Registros: 1
Tabla cargada: bridge_product_tags | Registros: 364


In [141]:
cursor = conn.cursor()

cursor.execute("""
    SELECT
        name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
""")

tables = cursor.fetchall()

for table in tables:
    print(table[0])

bridge_product_tags
dim_date
dim_product
dim_reviewer
dim_user
fact_cart_items
fact_carts
fact_product_reviews


In [142]:
for table_name, dataframe in gold_tables.items():

    cursor.execute(f"SELECT COUNT(*) FROM {table_name}")

    sqlite_count = cursor.fetchone()[0]
    pandas_count = len(dataframe)

    print(
        f"{table_name}: "
        f"Pandas = {pandas_count} | "
        f"SQLite = {sqlite_count} | "
        f"Coinciden = {pandas_count == sqlite_count}"
    )

fact_cart_items: Pandas = 800 | SQLite = 800 | Coinciden = True
fact_carts: Pandas = 208 | SQLite = 208 | Coinciden = True
fact_product_reviews: Pandas = 582 | SQLite = 582 | Coinciden = True
dim_product: Pandas = 194 | SQLite = 194 | Coinciden = True
dim_user: Pandas = 208 | SQLite = 208 | Coinciden = True
dim_reviewer: Pandas = 191 | SQLite = 191 | Coinciden = True
dim_date: Pandas = 1 | SQLite = 1 | Coinciden = True
bridge_product_tags: Pandas = 364 | SQLite = 364 | Coinciden = True


In [143]:
cursor.execute("""
    SELECT
        p.category,
        COUNT(*) AS total_cart_lines,
        SUM(f.quantity) AS total_units,
        ROUND(SUM(f.net_amount), 2) AS total_net_amount
    FROM fact_cart_items AS f
    INNER JOIN dim_product AS p
        ON f.product_id = p.product_id
    GROUP BY p.category
    ORDER BY total_net_amount DESC;
""")

results = cursor.fetchall()

for row in results:
    print(row)

('vehicle', 19, 60, 1682064.05)
('motorcycle', 23, 84, 609460.61)
('mens-watches', 26, 72, 593063.86)
('womens-watches', 18, 45, 249189.51)
('laptops', 16, 56, 85006.24)
('furniture', 22, 73, 77674.97)
('smartphones', 60, 177, 73130.95)
('mobile-accessories', 70, 198, 26160.99)
('tablets', 14, 40, 14890.16)
('mens-shoes', 23, 75, 8278.9)
('womens-bags', 21, 62, 5849.91)
('kitchen-accessories', 119, 361, 5714.33)
('fragrances', 24, 64, 4958.05)
('womens-dresses', 19, 49, 4064.13)
('sports-accessories', 60, 169, 3776.68)
('home-decoration', 28, 81, 2747.3)
('tops', 25, 77, 2233.24)
('groceries', 113, 354, 1881.5)
('womens-shoes', 15, 48, 1538.66)
('sunglasses', 17, 57, 1509.44)
('mens-shirts', 19, 57, 1357.63)
('womens-jewellery', 13, 42, 906.39)
('beauty', 23, 70, 821.84)
('skin-care', 13, 46, 430.24)


In [145]:
cursor.execute("""
    SELECT
        p.product_id,
        p.title,
        p.category,
        SUM(f.quantity) AS total_units,
        ROUND(SUM(f.gross_amount), 2) AS gross_amount,
        ROUND(SUM(f.discount_amount), 2) AS discount_amount,
        ROUND(SUM(f.net_amount), 2) AS net_amount
    FROM fact_cart_items AS f
    INNER JOIN dim_product AS p
        ON f.product_id = p.product_id
    WHERE p.category = 'vehicle'
    GROUP BY
        p.product_id,
        p.title,
        p.category
    ORDER BY net_amount DESC;
""")

results = cursor.fetchall()

for row in results:
    print(row)

(170, 'Durango SXT RWD', 'vehicle', 19, 702999.81, 115573.17, 587426.64)
(169, 'Dodge Hornet GT Plus', 'vehicle', 17, 424999.83, 11177.5, 413822.33)
(167, '300 Touring', 'vehicle', 9, 260999.91, 10387.8, 250612.11)
(171, 'Pacifica Touring', 'vehicle', 8, 255999.92, 37785.58, 218214.34)
(168, 'Charger SXT RWD', 'vehicle', 7, 230999.93, 19011.3, 211988.63)


In [146]:
cursor.execute("""
    SELECT
        p.product_id,
        p.title,
        SUM(f.quantity) AS total_units,
        ROUND(SUM(f.net_amount), 2) AS net_amount,
        ROUND(
            SUM(f.net_amount) * 100.0 /
            SUM(SUM(f.net_amount)) OVER (),
            2
        ) AS category_share_percentage
    FROM fact_cart_items AS f
    INNER JOIN dim_product AS p
        ON f.product_id = p.product_id
    WHERE p.category = 'vehicle'
    GROUP BY
        p.product_id,
        p.title
    ORDER BY net_amount DESC;
""")

for row in cursor.fetchall():
    print(row)

(170, 'Durango SXT RWD', 19, 587426.64, 34.92)
(169, 'Dodge Hornet GT Plus', 17, 413822.33, 24.6)
(167, '300 Touring', 9, 250612.11, 14.9)
(171, 'Pacifica Touring', 8, 218214.34, 12.97)
(168, 'Charger SXT RWD', 7, 211988.63, 12.6)


In [147]:
cursor.execute("""
    SELECT
        p.category,
        p.product_id,
        p.title,
        SUM(f.quantity) AS total_units,
        ROUND(SUM(f.net_amount), 2) AS net_amount,
        ROUND(
            SUM(f.net_amount) * 100.0 /
            SUM(SUM(f.net_amount)) OVER (
                PARTITION BY p.category
            ),
            2
        ) AS category_share_percentage
    FROM fact_cart_items AS f
    INNER JOIN dim_product AS p
        ON f.product_id = p.product_id
    GROUP BY
        p.category,
        p.product_id,
        p.title
    ORDER BY
        p.category,
        net_amount DESC;
""")

results = cursor.fetchall()

for row in results:
    print(row)

('beauty', 3, 'Powder Canister', 21, 283.8, 34.53)
('beauty', 2, 'Eyeshadow Palette with Mirror', 11, 179.88, 21.89)
('beauty', 1, 'Essence Mascara Lash Princess', 18, 160.99, 19.59)
('beauty', 4, 'Red Lipstick', 11, 125.51, 15.27)
('beauty', 5, 'Red Nail Polish', 9, 71.66, 8.72)
('fragrances', 7, 'Chanel Coco Noir Eau De', 16, 1736.46, 35.02)
('fragrances', 8, "Dior J'adore", 19, 1458.11, 29.41)
('fragrances', 9, 'Dolce Shine Eau de', 10, 695.56, 14.03)
('fragrances', 6, 'Calvin Klein CK One', 12, 588.56, 11.87)
('fragrances', 10, 'Gucci Bloom Eau de', 7, 479.36, 9.67)
('furniture', 12, 'Annibale Colombo Sofa', 23, 49219.81, 63.37)
('furniture', 14, 'Knoll Saarinen Executive Conference Chair', 22, 10778.68, 13.88)
('furniture', 11, 'Annibale Colombo Bed', 5, 8685.8, 11.18)
('furniture', 15, 'Wooden Bathroom Sink With Mirror', 7, 5107.13, 6.58)
('furniture', 13, 'Bedside Table African Cherry', 16, 3883.55, 5.0)
('groceries', 18, 'Cat Food', 37, 300.79, 15.99)
('groceries', 36, 'Protein

In [148]:
cursor.execute("""
    SELECT
        p.product_id,
        p.title,
        p.category,
        SUM(f.quantity) AS total_units,
        ROUND(SUM(f.net_amount), 2) AS net_amount
    FROM fact_cart_items AS f
    INNER JOIN dim_product AS p
        ON f.product_id = p.product_id
    GROUP BY
        p.product_id,
        p.title,
        p.category
    ORDER BY net_amount DESC
    LIMIT 10;
""")

results = cursor.fetchall()

for row in results:
    print(row)

(170, 'Durango SXT RWD', 'vehicle', 19, 587426.64)
(169, 'Dodge Hornet GT Plus', 'vehicle', 17, 413822.33)
(115, 'MotoGP CI.H1', 'motorcycle', 22, 307163.78)
(167, '300 Touring', 'vehicle', 9, 250612.11)
(98, 'Rolex Submariner Watch', 'mens-watches', 17, 225980.83)
(171, 'Pacifica Touring', 'vehicle', 8, 218214.34)
(168, 'Charger SXT RWD', 'vehicle', 7, 211988.63)
(97, 'Rolex Datejust', 'mens-watches', 14, 148255.66)
(117, 'Sportbike Motorcycle', 'motorcycle', 20, 132629.82)
(192, 'Rolex Datejust Women', 'womens-watches', 14, 129452.29)


In [149]:
cursor.execute("""
    SELECT
        p.product_id,
        p.title,
        p.category,
        SUM(f.quantity) AS total_units,
        ROUND(SUM(f.net_amount), 2) AS net_amount
    FROM fact_cart_items AS f
    INNER JOIN dim_product AS p
        ON f.product_id = p.product_id
    GROUP BY
        p.product_id,
        p.title,
        p.category
    ORDER BY total_units DESC
    LIMIT 10;
""")

results = cursor.fetchall()

for row in results:
    print(row)
    

(18, 'Cat Food', 'groceries', 37, 300.79)
(133, 'Samsung Galaxy S10', 'smartphones', 31, 20486.66)
(113, 'Generic Motorcycle', 'motorcycle', 28, 98447.75)
(46, 'Plant Pot', 'home-decoration', 27, 377.03)
(100, 'Apple Airpods', 'mobile-accessories', 27, 2964.33)
(164, 'Gray Dress', 'tops', 26, 779.82)
(28, 'Ice Cream', 'groceries', 25, 125.31)
(120, 'Vaseline Men Body and Face Lotion', 'skin-care', 25, 216.31)
(166, 'Tartan Dress', 'tops', 25, 870.27)
(43, 'Decoration Swing', 'home-decoration', 24, 1289.9)
